# 🚀 Qwen-Image 2.1 免 VAE 直出像素头（Direct Pixel Head）全云端一键训练与评测台（Colab L4 24GB 专用）

本 Notebook 全程运行于 **Google Colab L4 (24GB VRAM)**，**本机零算力负担**，一键完成全网首个 **`Qwen-Image 2.1` Direct RGBA Pixel Head** 的数据构建、离线特征缓存、双模式交替训练与三方跑分对比：

- **核心原理（`FastFrames` `arXiv:2611.23226`）**：冻结 `Qwen-Image 2.1` 主干，截获最后一步 `norm_out -> proj_out` 前的 `4096` 维上下文特征与 `64` 通道潜变量 `x_t`，用仅 **`~26.9M` 参数（约 `103 MB`）的轻量桥接 ConvNeXt + PixelShuffle 头** 直接输出 **4 通道 RGBA 像素**，彻底替代 `644 MB` 的重型 64 通道 VAE 解码器！
- **双模式 + 全场景通吃**：
  1. 训练时自动 **50% / 50% 交替抽样**「**8 步加速 LoRA 模式**（$\sigma \in [0.12, 0.30]$）」与「**20~35 步精修模式**（$\sigma \in [0.04, 0.12]$）」，单文件同时兼容开/关加速 LoRA；
  2. 天然同时兼容 **文生图 (T2I)** 与 **`<image1>` 多图参考指令编辑 (Omni Edit)**；
  3. 原生支持 **RGB 图像 + Alpha 透明通道（RGBA 贴纸/立绘抠图）** 双输出！
- **L4 24GB 显存严控**：编码缓存后彻底卸载 `10 GB` 文本编码器，正式训练阶段 **L4 峰值显存仅 ~10.8 GB / 22.5 GB**，稳如泰山。

### 一键运行顺序
右上角确认运行时为 **`Python 3` + `L4 GPU`** ➔ 直接点击顶部菜单 **「代码执行程序 (Runtime) ➔ 全部运行 (Run all)」**（或依次运行 **单元格 1 → 2 → 3 → 4 → 5**；单元格 6 为可选二阶 `2×` 超分头训练）。

In [ ]:
# @title 1. 挂载 Google Drive、校验 L4 显卡并自动部署开源代码与 ComfyUI 插件
import os
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)
drive.mount("/content/drive")

DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
PROJECT_DRIVE = Path("/content/drive/MyDrive/FastFrames_QwenImage21")
CKPT_DRIVE = PROJECT_DRIVE / "checkpoints"
EVAL_DRIVE = PROJECT_DRIVE / "eval_report"
PIXEL_HEADS_DRIVE = DRIVE_HUB / "models/pixel_heads"
VAE_DRIVE = DRIVE_HUB / "models/vae"
CUSTOM_NODE_DRIVE = DRIVE_HUB / "custom_nodes_qwenimage21/ComfyUI-QwenImage21-PixelHead"
WF_DRIVE_DIR = DRIVE_HUB / "user_qwenimage21/default/workflows"
LOCAL_SRC = Path("/content/ComfyUI-QwenImage21-PixelHead")

for d in (CKPT_DRIVE, EVAL_DRIVE, PIXEL_HEADS_DRIVE, VAE_DRIVE, CUSTOM_NODE_DRIVE, WF_DRIVE_DIR, LOCAL_SRC):
    d.mkdir(parents=True, exist_ok=True)

# 写入 100% Clean-Room MIT 插件与对比工作流（同时部署到当前训练环境与你的 QwenImage21 ComfyUI 目录）
FILES = {
    "qwen21_pixel_head.py": '"""Clean-room PyTorch implementation of Direct Pixel Readout Head for Qwen-Image 2.1.\n\nLicense: MIT\nArchitecture follows the public mathematical formulation of FastFrames (arXiv:2611.23226):\n  - Frozen Qwen-Image 2.1 DiT (`QwenImage21Transformer2DModel`, inner_dim=4096, latent_channels=64, 16x spatial).\n  - Pre-hook taps `f` (`[B, H_lat*W_lat, 4096]`, post-`norm_out`, pre-`proj_out`) plus the exact denoised\n    normalized latent `x0_hat_norm` (`[B, 64, H_lat, W_lat]`, sigma-invariant across all step counts).\n  - Feature LayerNorm + bridge linear (`4096 -> width`, bias=False) + latent linear (`64 -> width`, bias=True)\n    + entry LayerNorm at 16px token grid.\n  - `depth_low` ConvNeXt2d blocks at 16px token grid -> 2x PixelShuffle -> `depth_high` ConvNeXt2d blocks at 8px grid.\n  - Final LayerNorm + FP32 Linear (`width//2 -> 8*8*out_channels`) producing RGB(A) in `[-1, 1]`.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nTOKEN_PIXELS = 16  # Qwen-Image 2.1 VAE spatial downscale is 16x, DiT patch is 1x1\n\n\nclass ConvNeXtBlock2d(nn.Module):\n    """2D ConvNeXt block with 3x3 depthwise conv, channel LayerNorm, 4x GELU MLP, and LayerScale."""\n\n    def __init__(self, dim: int, mlp_ratio: int = 4, ls_init: float = 1.0):\n        super().__init__()\n        self.dw = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)\n        self.norm = nn.LayerNorm(dim, eps=1e-6)\n        hidden = dim * mlp_ratio\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, dim)\n        self.gamma = nn.Parameter(torch.full((dim,), float(ls_init)))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # x: [B, C, H, W]\n        h = self.dw(x).movedim(1, -1)  # [B, H, W, C]\n        h = self.fc2(F.gelu(self.fc1(self.norm(h))))\n        h = (h * self.gamma).movedim(-1, 1)\n        return x + h\n\n\ndef unpatchify_2d(tokens: torch.Tensor, patch: int, out_channels: int) -> torch.Tensor:\n    """Convert `[B, gh, gw, patch * patch * C]` -> `[B, C, gh * patch, gw * patch]`."""\n    b, gh, gw, _ = tokens.shape\n    x = tokens.view(b, gh, gw, patch, patch, out_channels)\n    x = x.permute(0, 5, 1, 3, 2, 4).contiguous()\n    return x.view(b, out_channels, gh * patch, gw * patch)\n\n\nclass QwenImage21PixelHead(nn.Module):\n    """1x Direct Pixel Head for Qwen-Image 2.1 (`16x16` RGBA/RGB pixels per DiT token)."""\n\n    def __init__(\n        self,\n        d_feat: int = 4096,\n        d_lat: int = 64,\n        width: int = 1024,\n        depth_low: int = 2,\n        depth_high: int = 1,\n        out_channels: int = 4,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        if width % 2 != 0:\n            raise ValueError(f"width must be even, got {width}")\n        self.d_feat = int(d_feat)\n        self.d_lat = int(d_lat)\n        self.width = int(width)\n        self.depth_low = int(depth_low)\n        self.depth_high = int(depth_high)\n        self.out_channels = int(out_channels)\n        self.ls_init = float(ls_init)\n\n        w_high = width // 2\n        self.out_patch = TOKEN_PIXELS // 2  # 8x8 pixels per 2x-upsampled position\n\n        # Normalize unbounded LastLayer AdaLN features before bridging into trunk width\n        self.feat_norm = nn.LayerNorm(self.d_feat, eps=1e-6)\n        self.feat_proj = nn.Linear(self.d_feat, width, bias=False)\n        self.lat_proj = nn.Linear(self.d_lat, width, bias=True)\n        self.entry_norm = nn.LayerNorm(width, eps=1e-6)\n\n        self.low_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(width, ls_init=ls_init) for _ in range(depth_low)]\n        )\n        self.up_norm = nn.LayerNorm(width, eps=1e-6)\n        self.up_proj = nn.Linear(width, 4 * w_high)\n        self.high_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_high, ls_init=ls_init) for _ in range(depth_high)]\n        )\n        self.norm_out = nn.LayerNorm(w_high, eps=1e-6)\n        self.out_proj = nn.Linear(w_high, self.out_patch * self.out_patch * self.out_channels)\n        # Keep initial pixel predictions well inside [-0.2, 0.2] so early gradients never clip/explode\n        nn.init.normal_(self.out_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out_proj.bias)\n\n        self.meta: dict[str, str] = {}\n\n    def forward_trunk(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        """Run shared trunk up to `high_blocks` (`[B, width//2, 2*gh, 2*gw]`).\n\n        Args:\n            feat: `[B, gh * gw, d_feat]` or `[B, gh, gw, d_feat]` pre-`proj_out` hidden states.\n            x0_hat_norm: `[B, d_lat, gh, gw]` denoised normalized latent at the final step.\n        """\n        b, _, gh, gw = x0_hat_norm.shape\n        if feat.ndim == 3:\n            feat = feat.view(b, gh, gw, self.d_feat)\n        lat_tok = x0_hat_norm.movedim(1, -1)  # [B, gh, gw, d_lat]\n\n        h = self.entry_norm(self.feat_proj(self.feat_norm(feat)) + self.lat_proj(lat_tok))\n        h = h.movedim(-1, 1).contiguous()  # [B, width, gh, gw]\n        for blk in self.low_blocks:\n            h = blk(h)\n\n        u = self.up_proj(self.up_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h = F.pixel_shuffle(u, 2)  # [B, width//2, 2*gh, 2*gw]\n        for blk in self.high_blocks:\n            h = blk(h)\n        return h\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        """Return `[B, out_channels, gh*16, gw*16]` in `[-1, 1]` (unclamped during training)."""\n        h = self.forward_trunk(feat, x0_hat_norm)\n        h_tok = self.norm_out(h.movedim(1, -1))  # [B, 2*gh, 2*gw, width//2]\n        # Keep final linear projection in fp32 to prevent 1-2 level quantization banding\n        with torch.autocast(device_type=h_tok.device.type, enabled=False):\n            patches = self.out_proj(h_tok.float())\n        return unpatchify_2d(patches, self.out_patch, self.out_channels)\n\n\nclass QwenImage21PixelHeadX2(nn.Module):\n    """Optional 2x Super-Resolution Direct Pixel Head (`32x32` pixels per DiT token)."""\n\n    def __init__(\n        self,\n        base: QwenImage21PixelHead,\n        depth_x2: int = 2,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        self.base = base\n        self.depth_x2 = int(depth_x2)\n        w_in = base.width // 2\n        w_x2 = w_in // 2\n        self.out_channels = base.out_channels\n        self.out_patch = base.out_patch  # 8px per 4x-upsampled grid cell = 32px per DiT token\n\n        self.up2_norm = nn.LayerNorm(w_in, eps=1e-6)\n        self.up2_proj = nn.Linear(w_in, 4 * w_x2)\n        self.x2_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_x2, ls_init=ls_init) for _ in range(depth_x2)]\n        )\n        self.norm2 = nn.LayerNorm(w_x2, eps=1e-6)\n        self.out2_proj = nn.Linear(w_x2, self.out_patch * self.out_patch * self.out_channels)\n        nn.init.normal_(self.out2_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out2_proj.bias)\n        self.meta: dict[str, str] = {}\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor, scale: int = 2) -> torch.Tensor:\n        h = self.base.forward_trunk(feat, x0_hat_norm)\n        if scale == 1:\n            h1 = self.base.norm_out(h.movedim(1, -1))\n            with torch.autocast(device_type=h1.device.type, enabled=False):\n                p1 = self.base.out_proj(h1.float())\n            return unpatchify_2d(p1, self.base.out_patch, self.out_channels)\n\n        u2 = self.up2_proj(self.up2_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        g = F.pixel_shuffle(u2, 2)  # [B, width//4, 4*gh, 4*gw]\n        for blk in self.x2_blocks:\n            g = blk(g)\n        h2 = self.norm2(g.movedim(1, -1))\n        with torch.autocast(device_type=h2.device.type, enabled=False):\n            p2 = self.out2_proj(h2.float())\n        return unpatchify_2d(p2, self.out_patch, self.out_channels)\n\n\ndef save_qwen21_pixel_head(\n    head: QwenImage21PixelHead | QwenImage21PixelHeadX2,\n    path: str | Path,\n    extra_meta: Optional[dict[str, str]] = None,\n) -> None:\n    from safetensors.torch import save_file\n\n    is_x2 = isinstance(head, QwenImage21PixelHeadX2)\n    base = head.base if is_x2 else head\n    meta = {\n        "arch": "qwen_image21_pixel_head_x2" if is_x2 else "qwen_image21_pixel_head",\n        "d_feat": str(base.d_feat),\n        "d_lat": str(base.d_lat),\n        "width": str(base.width),\n        "depth_low": str(base.depth_low),\n        "depth_high": str(base.depth_high),\n        "out_channels": str(base.out_channels),\n        "ls_init": str(base.ls_init),\n        "scale": "2" if is_x2 else "1",\n        "latent_input": "x0_hat_norm",\n    }\n    if is_x2:\n        meta["depth_x2"] = str(head.depth_x2)\n    if extra_meta:\n        for k, v in extra_meta.items():\n            meta[str(k)] = str(v)\n\n    sd = {k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    save_file(sd, str(path), metadata=meta)\n\n\ndef load_qwen21_pixel_head(\n    path: str | Path,\n    device: str | torch.device = "cpu",\n    dtype: torch.dtype = torch.float32,\n) -> QwenImage21PixelHead | QwenImage21PixelHeadX2:\n    from safetensors import safe_open\n    from safetensors.torch import load_file\n\n    path = Path(path)\n    with safe_open(str(path), framework="pt", device="cpu") as fh:\n        meta = dict(fh.metadata() or {})\n\n    arch = meta.get("arch", "qwen_image21_pixel_head")\n    base = QwenImage21PixelHead(\n        d_feat=int(meta.get("d_feat", "4096")),\n        d_lat=int(meta.get("d_lat", "64")),\n        width=int(meta.get("width", "1024")),\n        depth_low=int(meta.get("depth_low", "2")),\n        depth_high=int(meta.get("depth_high", "1")),\n        out_channels=int(meta.get("out_channels", "4")),\n        ls_init=float(meta.get("ls_init", "1.0")),\n    )\n    if arch == "qwen_image21_pixel_head_x2" or "depth_x2" in meta:\n        model: nn.Module = QwenImage21PixelHeadX2(\n            base=base,\n            depth_x2=int(meta.get("depth_x2", "2")),\n            ls_init=float(meta.get("ls_init", "1.0")),\n        )\n    else:\n        model = base\n\n    sd = load_file(str(path), device="cpu")\n    model.load_state_dict(sd, strict=True)\n    model = model.to(device=device, dtype=dtype)\n    # Keep final RGB(A) projection strictly in fp32 even if trunk runs in bf16\n    if isinstance(model, QwenImage21PixelHeadX2):\n        model.base.out_proj.float()\n        model.out2_proj.float()\n    else:\n        model.out_proj.float()\n    model.meta = meta\n    model.eval()\n    return model\n',
    "tap.py": '"""Non-invasive ComfyUI ModelPatcher tap for Qwen-Image 2.1 (`QwenImage21Transformer2DModel`).\n\nLicense: MIT\nCaptures the final denoising step\'s pre-`proj_out` normalized hidden states `f` (`[B, H*W, 4096]`)\nand the exact CFG-combined denoised normalized latent `x0_hat_norm` (`[B, 64, H_lat, W_lat]`), supporting:\n  - CFG = 1.0 (8-step Turbo / distilled) AND CFG > 1.0 (`f_u + cfg * (f_c - f_u)`)\n  - `QwenImage21Cache` KV prefix caching ON or OFF\n  - Both Text-to-Image (T2I) and Multi-Image Reference Editing (`<image1>`, `<image2>`, ...)\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport comfy.patcher_extension\nimport torch\n\n\nTAP_WRAPPER_KEY = "qwen21_pixel_head_tap"\n\n\n@dataclass\nclass StepRecord:\n    sigma: float\n    forwards: list[dict] = field(default_factory=list)\n    cond_scale: Optional[float] = None\n    denoised: Optional[torch.Tensor] = None\n\n\nclass Qwen21TapState:\n    """Stores only the latest actual DiT forward step\'s features on CPU."""\n\n    def __init__(self):\n        self.current_step: Optional[StepRecord] = None\n        self.last_valid_step: Optional[StepRecord] = None\n\n    def open_step(self, sigma: float) -> None:\n        self.current_step = StepRecord(sigma=sigma)\n\n    def close_step(self) -> None:\n        if self.current_step is not None and len(self.current_step.forwards) > 0:\n            self.last_valid_step = self.current_step\n        self.current_step = None\n\n    def resolve_final_features(self) -> tuple[torch.Tensor, torch.Tensor, float]:\n        """Return `(f_combined [B, H*W, 4096], x0_hat_norm [B, 64, H_lat, W_lat], sigma)`."""\n        step = self.last_valid_step\n        if step is None or not step.forwards:\n            raise RuntimeError(\n                "QwenImage21PixelDecode: 未捕获到任何前向特征！请确认已将 `QwenImage21 Pixel Head Tap` "\n                "串联在进入 KSampler 的 MODEL 连线上，且本次运行实际执行了采样步骤。"\n            )\n\n        # Group recorded chunks by ComfyUI\'s cond_or_uncond tag (0 = positive/cond, 1 = negative/uncond)\n        by_tag: dict[int, list[tuple[torch.Tensor, torch.Tensor]]] = {0: [], 1: []}\n        for rec in step.forwards:\n            cou = rec["cond_or_uncond"]\n            n_chunks = len(cou)\n            f_chunks = rec["f"].chunk(n_chunks, dim=0)\n            d_chunks = rec["denoised"].chunk(n_chunks, dim=0)\n            for tag, fc, dc in zip(cou, f_chunks, d_chunks):\n                by_tag.setdefault(int(tag), []).append((fc, dc))\n\n        if not by_tag.get(0):\n            raise RuntimeError("QwenImage21PixelDecode: 最后一步未找到正向条件 (cond=0) 特征记录。")\n\n        f_cond = torch.cat([p[0] for p in by_tag[0]], dim=0).float()\n        d_cond = torch.cat([p[1] for p in by_tag[0]], dim=0).float()\n\n        if by_tag.get(1) and step.cond_scale is not None and abs(step.cond_scale - 1.0) > 1e-5:\n            f_uncond = torch.cat([p[0] for p in by_tag[1]], dim=0).float()\n            d_uncond = torch.cat([p[1] for p in by_tag[1]], dim=0).float()\n            scale = float(step.cond_scale)\n            f_final = f_uncond + scale * (f_cond - f_uncond)\n            x0_hat_fallback = d_uncond + scale * (d_cond - d_uncond)\n        else:\n            f_final = f_cond\n            x0_hat_fallback = d_cond\n\n        x0_hat_norm = step.denoised.float() if step.denoised is not None else x0_hat_fallback\n        return f_final, x0_hat_norm, float(step.sigma)\n\n\ndef attach_qwen21_tap(model_patcher) -> tuple[object, Qwen21TapState]:\n    """Clone `model_patcher` and install lightweight step + `proj_out` pre-hooks."""\n    cloned = model_patcher.clone()\n    state = Qwen21TapState()\n\n    # Remove any prior tap hooks if re-attached\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.CALC_COND_BATCH, TAP_WRAPPER_KEY)\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.APPLY_MODEL, TAP_WRAPPER_KEY)\n\n    def calc_cond_batch_wrapper(executor, model, conds, x_in, timestep, model_options):\n        sig = float(timestep.flatten()[0].detach().cpu().item())\n        state.open_step(sig)\n        try:\n            return executor(model, conds, x_in, timestep, model_options)\n        finally:\n            state.close_step()\n\n    def apply_model_wrapper(executor, x, t, c_concat=None, c_crossattn=None, control=None, transformer_options={}, **kwargs):\n        step = state.current_step\n        cou = (transformer_options or {}).get("cond_or_uncond")\n        if step is None or not cou:\n            return executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n\n        base_model = executor.class_obj\n        diff_model = getattr(base_model, "diffusion_model", None)\n        proj_out = getattr(diff_model, "proj_out", None)\n        if proj_out is None:\n            raise RuntimeError(\n                f"QwenImage21PixelHeadTap: 当前模型 ({type(diff_model).__name__}) 不包含 `proj_out`，请确认底模为 Qwen-Image 2.1。"\n            )\n\n        rec = {\n            "denoised": None,\n            "f": None,\n            "cond_or_uncond": list(cou),\n        }\n\n        def _pre_proj_hook(_mod, inputs):\n            if rec["f"] is None and len(inputs) > 0:\n                rec["f"] = inputs[0].detach().to(device="cpu", dtype=torch.bfloat16)\n\n        handle = proj_out.register_forward_pre_hook(_pre_proj_hook)\n        try:\n            # BaseModel.apply_model returns calculate_denoised(sigma, model_output, x) = x0_hat_norm\n            out = executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n        finally:\n            handle.remove()\n\n        if rec["f"] is not None:\n            rec["denoised"] = out.detach().to(device="cpu", dtype=torch.float32)\n            step.forwards.append(rec)\n        return out\n\n    def post_cfg_hook(args):\n        # Note: in comfy/samplers.py, `cfg_function` runs right AFTER `calc_cond_batch` returns,\n        # so `state.last_valid_step` holds the step that just finished!\n        target_step = state.current_step if state.current_step is not None else state.last_valid_step\n        if target_step is not None:\n            if "cond_scale" in args:\n                target_step.cond_scale = float(args["cond_scale"])\n            if "denoised" in args and args["denoised"] is not None:\n                target_step.denoised = args["denoised"].detach().to(device="cpu", dtype=torch.float32)\n        return args["denoised"]\n\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.CALC_COND_BATCH,\n        TAP_WRAPPER_KEY,\n        calc_cond_batch_wrapper,\n    )\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.APPLY_MODEL,\n        TAP_WRAPPER_KEY,\n        apply_model_wrapper,\n    )\n    cloned.set_model_sampler_post_cfg_function(post_cfg_hook)\n    cloned.model_options[TAP_WRAPPER_KEY] = state\n    return cloned, state\n',
    "nodes.py": '"""ComfyUI Nodes for Qwen-Image 2.1 Direct Pixel Head (Zero-VAE RGB & RGBA Decoding).\n\nLicense: MIT\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport time\nimport logging\n\nimport folder_paths\nimport comfy.model_management\nimport torch\n\nfrom .qwen21_pixel_head import (\n    QwenImage21PixelHeadX2,\n    load_qwen21_pixel_head,\n)\nfrom .tap import attach_qwen21_tap\n\nlogger = logging.getLogger(__name__)\n\n# Register dedicated `models/pixel_heads` directory AND allow loading from `models/vae` for convenience\nPIXEL_HEAD_DIR = os.path.join(folder_paths.models_dir, "pixel_heads")\nos.makedirs(PIXEL_HEAD_DIR, exist_ok=True)\nif "pixel_heads" not in folder_paths.folder_names_and_paths:\n    folder_paths.add_model_folder_path("pixel_heads", PIXEL_HEAD_DIR)\n\n\ndef _list_candidate_files() -> list[str]:\n    seen = set()\n    items = []\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            for fn in folder_paths.get_filename_list(folder_key):\n                if fn.endswith(".safetensors") and fn not in seen:\n                    seen.add(fn)\n                    items.append(fn)\n    return sorted(items) if items else ["qwen_image_2.1_pixel_head.safetensors"]\n\n\ndef _resolve_head_path(name: str) -> str:\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            p = folder_paths.get_full_path(folder_key, name)\n            if p and os.path.exists(p):\n                return p\n    raise FileNotFoundError(\n        f"未找到 Pixel Head 权重 `{name}`，请将其放入 `ComfyUI/models/pixel_heads/` 或 `ComfyUI/models/vae/`。"\n    )\n\n\nclass QwenImage21PixelHeadLoader:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "head_name": (_list_candidate_files(),),\n                "precision": (["bf16_trunk_fp32_out", "fp32"], {"default": "bf16_trunk_fp32_out"}),\n            }\n        }\n\n    RETURN_TYPES = ("QWEN21_PIXEL_HEAD",)\n    RETURN_NAMES = ("pixel_head",)\n    FUNCTION = "load_head"\n    CATEGORY = "advanced/qwen_image21"\n    TITLE = "Load Qwen-Image 2.1 Pixel Head"\n\n    def load_head(self, head_name: str, precision: str):\n        path = _resolve_head_path(head_name)\n        dtype = torch.bfloat16 if precision == "bf16_trunk_fp32_out" else torch.float32\n        head = load_qwen21_pixel_head(path, device="cpu", dtype=dtype)\n        return (head,)\n\n\nclass QwenImage21PixelHeadTap:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "model": ("MODEL",),\n            }\n        }\n\n    RETURN_TYPES = ("MODEL", "QWEN21_TAP")\n    RETURN_NAMES = ("model", "pixel_tap")\n    FUNCTION = "install_tap"\n    CATEGORY = "advanced/qwen_image21"\n    TITLE = "Qwen-Image 2.1 Pixel Head Tap"\n\n    def install_tap(self, model):\n        patched_model, tap_state = attach_qwen21_tap(model)\n        return (patched_model, tap_state)\n\n\nclass QwenImage21PixelDecode:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "samples": ("LATENT",),\n                "pixel_head": ("QWEN21_PIXEL_HEAD",),\n                "pixel_tap": ("QWEN21_TAP",),\n                "output_scale": (["auto", "1x", "2x"], {"default": "auto"}),\n            }\n        }\n\n    RETURN_TYPES = ("IMAGE", "MASK", "IMAGE")\n    RETURN_NAMES = ("image_rgb", "alpha_mask", "image_rgba")\n    FUNCTION = "decode"\n    CATEGORY = "advanced/qwen_image21"\n    TITLE = "Qwen-Image 2.1 Pixel Decode (Zero VAE)"\n\n    @torch.no_grad()\n    def decode(self, samples, pixel_head, pixel_tap, output_scale: str = "auto"):\n        # `samples` dependency guarantees ComfyUI executes KSampler BEFORE this decode node!\n        _ = samples["samples"]\n        f_cpu, xt_cpu, sigma = pixel_tap.resolve_final_features()\n\n        device = comfy.model_management.get_torch_device()\n        is_x2 = isinstance(pixel_head, QwenImage21PixelHeadX2)\n        if output_scale == "auto":\n            scale = 2 if is_x2 else 1\n        elif output_scale == "2x":\n            if not is_x2:\n                raise ValueError("当前加载的是 1x Pixel Head 权重，若需 2x 直出请加载带 `_x2` 的二阶权重。")\n            scale = 2\n        else:\n            scale = 1\n\n        head_dev = pixel_head.to(device)\n        trunk_dtype = next(head_dev.parameters()).dtype\n        f_dev = f_cpu.to(device=device, dtype=trunk_dtype)\n        xt_dev = xt_cpu.to(device=device, dtype=trunk_dtype)\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        t0 = time.perf_counter()\n\n        if is_x2:\n            out_nchw = head_dev(f_dev, xt_dev, scale=scale)\n        else:\n            out_nchw = head_dev(f_dev, xt_dev)\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        dt_ms = (time.perf_counter() - t0) * 1000.0\n\n        # Convert [-1, 1] NCHW -> [0, 1] NHWC ComfyUI IMAGE format\n        out_nhwc = ((out_nchw.float() + 1.0) * 0.5).clamp_(0.0, 1.0).movedim(1, -1).cpu()\n        # Return head to CPU so idle VRAM stays zero\n        pixel_head.to("cpu")\n\n        if out_nhwc.shape[-1] == 4:\n            rgb = out_nhwc[..., :3].contiguous()\n            alpha = out_nhwc[..., 3].contiguous()\n            rgba = out_nhwc.contiguous()\n        else:\n            rgb = out_nhwc[..., :3].contiguous()\n            alpha = torch.ones(rgb.shape[:-1], dtype=torch.float32)\n            rgba = torch.cat([rgb, alpha.unsqueeze(-1)], dim=-1)\n\n        logger.info(\n            f"[Qwen21PixelDecode] Decoded {tuple(rgb.shape[1:3])} (scale={scale}x, last_sigma={sigma:.4f}) in {dt_ms:.2f} ms (Zero VAE)"\n        )\n        return (rgb, alpha, rgba)\n\n\nNODE_CLASS_MAPPINGS = {\n    "QwenImage21PixelHeadLoader": QwenImage21PixelHeadLoader,\n    "QwenImage21PixelHeadTap": QwenImage21PixelHeadTap,\n    "QwenImage21PixelDecode": QwenImage21PixelDecode,\n}\n\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "QwenImage21PixelHeadLoader": "Load Qwen-Image 2.1 Pixel Head",\n    "QwenImage21PixelHeadTap": "Qwen-Image 2.1 Pixel Head Tap",\n    "QwenImage21PixelDecode": "Qwen-Image 2.1 Pixel Decode (Zero VAE)",\n}\n',
    "__init__.py": 'from .nodes import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS\n\n__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS"]\n',
    "README.md": '# ComfyUI-QwenImage21-PixelHead\n\nZero-VAE Direct Pixel Readout Head (`RGB` + `Alpha / RGBA`, `1×` & `2×`) for **Qwen-Image 2.1** (`Qwen/Qwen-Image-2.1`, `Qwen-Image-2.1-Omni`, and `8-Step` / `Turbo` variants) in ComfyUI.\n\nInspired by *FastFrames: Video Generation at Interactive Latency* (`arXiv:2611.23226`), this node pack taps the pre-`proj_out` contextualized `4096`-dim DiT features and `64`-ch `x_t` latent at the final denoising step, decoding directly to pixels in **~4–8 ms** without loading the `644 MB` 64-channel Qwen-Image 2.1 VAE decoder.\n\n## Nodes\n\n1. **`Load Qwen-Image 2.1 Pixel Head`** (`QwenImage21PixelHeadLoader`)\n   - Loads `qwen_image_2.1_pixel_head.safetensors` (`1×`) or `qwen_image_2.1_pixel_head_x2.safetensors` (`2×`) from `ComfyUI/models/pixel_heads/` or `ComfyUI/models/vae/`.\n2. **`Qwen-Image 2.1 Pixel Head Tap`** (`QwenImage21PixelHeadTap`)\n   - Insert on the `MODEL` wire right before `KSampler` / `SamplerCustomAdvanced`.\n   - Compatible with `QwenImage21Cache` (`auto` / `gpu` / `cpu` / `off`), `CFG = 1.0` (8-step distilled) and `CFG > 1.0` (Base), as well as both **Text-to-Image (T2I)** and **Multi-Image Reference Editing (`<image1>`, `<image2>`)**.\n3. **`Qwen-Image 2.1 Pixel Decode (Zero VAE)`** (`QwenImage21PixelDecode`)\n   - Replaces `VAEDecode`. Outputs `image_rgb` (`[B, H, W, 3]`), `alpha_mask` (`[B, H, W]`), and `image_rgba` (`[B, H, W, 4]`).\n\n## License\n\nMIT License (100% clean-room implementation).\n',
    "LICENSE": 'MIT License\n\nCopyright (c) 2026\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n',
}
for fname, content in FILES.items():
    (LOCAL_SRC / fname).write_text(content, encoding="utf-8")
    (CUSTOM_NODE_DRIVE / fname).write_text(content, encoding="utf-8")

(WF_DRIVE_DIR / "03_QwenImage21_PixelHead_免VAE极速直出与对比.json").write_text('{\n  "last_node_id": 15,\n  "last_link_id": 15,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "MarkdownNote",\n      "pos": [\n        40,\n        -250\n      ],\n      "size": [\n        1520,\n        200\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "title": "⚡ Qwen-Image 2.1 PixelHead：免 VAE 毫秒级直出与原版 VAE 同屏对比（Colab L4 专用）",\n      "properties": {\n        "Node name for S&R": "MarkdownNote",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "### Qwen-Image 2.1 Direct Pixel Head 实测与同屏对比工作流（Colab L4 24GB）\\n- **默认极速直出（零 VAE 显存）**：`KSampler` 采样结束后，直接由 `7. Qwen-Image 2.1 Pixel Decode (Zero VAE)` 在 **~5 ms** 内把末步 4096 维隐藏态还原为高清 RGB 与 Alpha 透明通道！\\n- **同屏画质对比（可选）**：下方紫色的 `9. 原版 VAE 加载`、`10. 原版 VAE 解码`、`11. 原版 VAE 保存` 默认处于 Bypass 关闭状态；选中它们按 **`Ctrl + B`** 开启，即可在同一随机种子下 1:1 对比 Pixel Head 与 644MB 官方 VAE 的细节表现！\\n- **全模式通用**：支持开启/关闭 `8Steps LoRA`（开=8步 CFG 1.0；关=25步 CFG 3.5），也支持在 `TextEncodeQwenImage21` 接入参考图做 `<image1>` 指令编辑。"\n      ],\n      "color": "#1e2d24",\n      "bgcolor": "#14221a"\n    },\n    {\n      "id": 2,\n      "type": "UNETLoader",\n      "pos": [\n        40,\n        20\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "flags": {},\n      "order": 1,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "slot_index": 0,\n          "links": [\n            1\n          ]\n        }\n      ],\n      "title": "1. 加载 Qwen-Image 2.1 主模型",\n      "properties": {\n        "Node name for S&R": "UNETLoader",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "qwen_image_2.1_int8_convrot.safetensors",\n        "default"\n      ],\n      "color": "#323",\n      "bgcolor": "#435"\n    },\n    {\n      "id": 3,\n      "type": "CLIPLoader",\n      "pos": [\n        40,\n        145\n      ],\n      "size": [\n        430,\n        98\n      ],\n      "flags": {},\n      "order": 2,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "slot_index": 0,\n          "links": [\n            4\n          ]\n        }\n      ],\n      "title": "2. 加载官方 Qwen3-VL-8B 编码器",\n      "properties": {\n        "Node name for S&R": "CLIPLoader",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "qwen3vl_8b_int8_convrot.safetensors",\n        "qwen_image",\n        "default"\n      ],\n      "color": "#323",\n      "bgcolor": "#435"\n    },\n    {\n      "id": 4,\n      "type": "LoraLoaderModelOnly",\n      "pos": [\n        40,\n        280\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "flags": {},\n      "order": 3,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "slot_index": 0,\n          "links": [\n            2\n          ]\n        }\n      ],\n      "title": "3. 8步加速 LoRA（默认开启=8步；Ctrl+B 关闭改 25 步）",\n      "properties": {\n        "Node name for S&R": "LoraLoaderModelOnly",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors",\n        1.0\n      ],\n      "color": "#323",\n      "bgcolor": "#435"\n    },\n    {\n      "id": 5,\n      "type": "QwenImage21Cache",\n      "pos": [\n        40,\n        405\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "flags": {},\n      "order": 4,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "slot_index": 0,\n          "links": [\n            3\n          ]\n        }\n      ],\n      "title": "4. KV 前缀加速缓存",\n      "properties": {\n        "Node name for S&R": "QwenImage21Cache",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "auto",\n        "default"\n      ],\n      "color": "#323",\n      "bgcolor": "#435"\n    },\n    {\n      "id": 6,\n      "type": "QwenImage21PixelHeadLoader",\n      "pos": [\n        40,\n        535\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "flags": {},\n      "order": 5,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "pixel_head",\n          "type": "QWEN21_PIXEL_HEAD",\n          "slot_index": 0,\n          "links": [\n            10\n          ]\n        }\n      ],\n      "title": "5. 加载训练好的 Qwen-Image 2.1 Pixel Head (~103MB)",\n      "properties": {\n        "Node name for S&R": "QwenImage21PixelHeadLoader"\n      },\n      "widgets_values": [\n        "qwen_image_2.1_pixel_head.safetensors",\n        "bf16_trunk_fp32_out"\n      ],\n      "color": "#1e2d24",\n      "bgcolor": "#264030"\n    },\n    {\n      "id": 7,\n      "type": "QwenImage21PixelHeadTap",\n      "pos": [\n        40,\n        660\n      ],\n      "size": [\n        430,\n        68\n      ],\n      "flags": {},\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 3\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "slot_index": 0,\n          "links": [\n            5\n          ]\n        },\n        {\n          "name": "pixel_tap",\n          "type": "QWEN21_TAP",\n          "slot_index": 1,\n          "links": [\n            11\n          ]\n        }\n      ],\n      "title": "6. 挂载末步特征截获钩子 (Pixel Head Tap)",\n      "properties": {\n        "Node name for S&R": "QwenImage21PixelHeadTap"\n      },\n      "color": "#1e2d24",\n      "bgcolor": "#264030"\n    },\n    {\n      "id": 8,\n      "type": "TextEncodeQwenImage21",\n      "pos": [\n        510,\n        20\n      ],\n      "size": [\n        450,\n        300\n      ],\n      "flags": {},\n      "order": 7,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 4\n        },\n        {\n          "name": "images.image_1",\n          "shape": 7,\n          "type": "IMAGE",\n          "link": null\n        },\n        {\n          "name": "vae",\n          "shape": 7,\n          "type": "VAE",\n          "link": null\n        }\n      ],\n      "outputs": [\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "slot_index": 0,\n          "links": [\n            6\n          ]\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "slot_index": 1,\n          "links": [\n            7\n          ]\n        },\n        {\n          "name": "latent",\n          "type": "LATENT",\n          "slot_index": 2,\n          "links": []\n        }\n      ],\n      "title": "7. 提示词与可选多图编辑条件编码",\n      "properties": {\n        "Node name for S&R": "TextEncodeQwenImage21",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "高质量日本 2D 二次元插画，赛璐璐平涂上色与锐利手绘黑色线稿。一位黑发少年坐在阳光洒入的卧室床沿，手握透明矿泉水瓶，衣褶清晰，画面干净细腻。",\n        "",\n        1024\n      ],\n      "color": "#232",\n      "bgcolor": "#353"\n    },\n    {\n      "id": 9,\n      "type": "EmptyLatentImage",\n      "pos": [\n        510,\n        360\n      ],\n      "size": [\n        450,\n        108\n      ],\n      "flags": {},\n      "order": 8,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "slot_index": 0,\n          "links": [\n            8\n          ]\n        }\n      ],\n      "title": "8. 画幅尺寸（32 的倍数，如 1024×1024 / 896×1152）",\n      "properties": {\n        "Node name for S&R": "EmptyLatentImage",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        1024,\n        1024,\n        1\n      ],\n      "color": "#223",\n      "bgcolor": "#2a3b4c"\n    },\n    {\n      "id": 10,\n      "type": "KSampler",\n      "pos": [\n        1000,\n        20\n      ],\n      "size": [\n        340,\n        265\n      ],\n      "flags": {},\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 5\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 6\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 7\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 8\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "slot_index": 0,\n          "links": [\n            9,\n            13\n          ]\n        }\n      ],\n      "title": "9. KSampler（8步 euler/simple CFG 1.0）",\n      "properties": {\n        "Node name for S&R": "KSampler",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        20261122,\n        "fixed",\n        8,\n        1.0,\n        "euler",\n        "simple",\n        1.0\n      ],\n      "color": "#223",\n      "bgcolor": "#2a3b4c"\n    },\n    {\n      "id": 11,\n      "type": "QwenImage21PixelDecode",\n      "pos": [\n        1000,\n        330\n      ],\n      "size": [\n        340,\n        125\n      ],\n      "flags": {},\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 9\n        },\n        {\n          "name": "pixel_head",\n          "type": "QWEN21_PIXEL_HEAD",\n          "link": 10\n        },\n        {\n          "name": "pixel_tap",\n          "type": "QWEN21_TAP",\n          "link": 11\n        }\n      ],\n      "outputs": [\n        {\n          "name": "image_rgb",\n          "type": "IMAGE",\n          "slot_index": 0,\n          "links": [\n            12\n          ]\n        },\n        {\n          "name": "alpha_mask",\n          "type": "MASK",\n          "slot_index": 1,\n          "links": []\n        },\n        {\n          "name": "image_rgba",\n          "type": "IMAGE",\n          "slot_index": 2,\n          "links": []\n        }\n      ],\n      "title": "10. Pixel Head 极速免 VAE 解码 (~5ms 直出 RGB + Alpha)",\n      "properties": {\n        "Node name for S&R": "QwenImage21PixelDecode"\n      },\n      "widgets_values": [\n        "auto"\n      ],\n      "color": "#1e2d24",\n      "bgcolor": "#264030"\n    },\n    {\n      "id": 12,\n      "type": "SaveImage",\n      "pos": [\n        1380,\n        20\n      ],\n      "size": [\n        440,\n        435\n      ],\n      "flags": {},\n      "order": 11,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 12\n        }\n      ],\n      "outputs": [],\n      "title": "11. 保存 Pixel Head 直出成图",\n      "properties": {\n        "Node name for S&R": "SaveImage",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "Qwen21_PixelHead_ZeroVAE"\n      ],\n      "color": "#1e2d24",\n      "bgcolor": "#264030"\n    },\n    {\n      "id": 13,\n      "type": "VAELoader",\n      "pos": [\n        510,\n        525\n      ],\n      "size": [\n        450,\n        68\n      ],\n      "flags": {},\n      "order": 12,\n      "mode": 4,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "slot_index": 0,\n          "links": [\n            14\n          ]\n        }\n      ],\n      "title": "12. [对比用] 原版 VAE（默认 Bypass，按 Ctrl+B 开启对比）",\n      "properties": {\n        "Node name for S&R": "VAELoader",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "qwen_image_2.1_vae_bf16.safetensors"\n      ],\n      "color": "#323",\n      "bgcolor": "#435"\n    },\n    {\n      "id": 14,\n      "type": "VAEDecode",\n      "pos": [\n        1000,\n        525\n      ],\n      "size": [\n        340,\n        68\n      ],\n      "flags": {},\n      "order": 13,\n      "mode": 4,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 13\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 14\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "slot_index": 0,\n          "links": [\n            15\n          ]\n        }\n      ],\n      "title": "13. [对比用] 原版 VAE 解码（默认 Bypass，Ctrl+B 开启）",\n      "properties": {\n        "Node name for S&R": "VAEDecode",\n        "cnr_id": "comfy-core"\n      }\n    },\n    {\n      "id": 15,\n      "type": "SaveImage",\n      "pos": [\n        1380,\n        500\n      ],\n      "size": [\n        440,\n        420\n      ],\n      "flags": {},\n      "order": 14,\n      "mode": 4,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 15\n        }\n      ],\n      "outputs": [],\n      "title": "14. [对比用] 原版 VAE 成图（默认 Bypass，Ctrl+B 开启）",\n      "properties": {\n        "Node name for S&R": "SaveImage",\n        "cnr_id": "comfy-core"\n      },\n      "widgets_values": [\n        "Qwen21_StockVAE_Ref"\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      2,\n      0,\n      4,\n      0,\n      "MODEL"\n    ],\n    [\n      2,\n      4,\n      0,\n      5,\n      0,\n      "MODEL"\n    ],\n    [\n      3,\n      5,\n      0,\n      7,\n      0,\n      "MODEL"\n    ],\n    [\n      4,\n      3,\n      0,\n      8,\n      0,\n      "CLIP"\n    ],\n    [\n      5,\n      7,\n      0,\n      10,\n      0,\n      "MODEL"\n    ],\n    [\n      6,\n      8,\n      0,\n      10,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      7,\n      8,\n      1,\n      10,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      8,\n      9,\n      0,\n      10,\n      3,\n      "LATENT"\n    ],\n    [\n      9,\n      10,\n      0,\n      11,\n      0,\n      "LATENT"\n    ],\n    [\n      10,\n      6,\n      0,\n      11,\n      1,\n      "QWEN21_PIXEL_HEAD"\n    ],\n    [\n      11,\n      7,\n      1,\n      11,\n      2,\n      "QWEN21_TAP"\n    ],\n    [\n      12,\n      11,\n      0,\n      12,\n      0,\n      "IMAGE"\n    ],\n    [\n      13,\n      10,\n      0,\n      14,\n      0,\n      "LATENT"\n    ],\n    [\n      14,\n      13,\n      0,\n      14,\n      1,\n      "VAE"\n    ],\n    [\n      15,\n      14,\n      0,\n      15,\n      0,\n      "IMAGE"\n    ]\n  ],\n  "groups": [],\n  "config": {},\n  "extra": {\n    "ds": {\n      "scale": 0.78,\n      "offset": [\n        80,\n        320\n      ]\n    }\n  },\n  "version": 0.4\n}', encoding="utf-8")

print("✅ Google Drive 挂载成功，插件与 03 对比工作流已同步至:", CUSTOM_NODE_DRIVE)

# 核查网盘已有 Qwen-Image 2.1 核心四件套状态
REQUIRED = [
    ("models/diffusion_models", "qwen_image_2.1_int8_convrot.safetensors"),
    ("models/text_encoders", "qwen3vl_8b_int8_convrot.safetensors"),
    ("models/vae", "qwen_image_2.1_vae_bf16.safetensors"),
    ("models/loras", "p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors"),
]
for sub, fn in REQUIRED:
    p = DRIVE_HUB / sub / fn
    status = f"✅ 已就绪 ({p.stat().st_size / 1e9:.2f} GB)" if p.exists() else "⬇️ 将在单元格 2 自动高速补齐"
    print(f"   - {sub}/{fn}: {status}")

In [ ]:
# @title 2. 初始化 ComfyUI 核心引擎并将模型高速预热至本地 NVMe SSD（杜绝云盘读取卡顿）
import json
import os
import shutil
import struct
import subprocess
import sys
import time
from pathlib import Path

t0 = time.time()
DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
COMFY_DIR = Path("/content/ComfyUI")
LOCAL_MODELS = Path("/content/local_models")
LOCAL_MODELS.mkdir(parents=True, exist_ok=True)

# @markdown ### 主模型选择（Teacher VAE 已固定为官方原版 `qwen_image_2.1_vae_bf16.safetensors`）：
# @markdown - `"official_qwen21_int8"`：直接使用你网盘已有的 `qwen-image-2.1-step90000-INT8-ConvRot`（通用 + 无审查兼容）
# @markdown - `"official_base_int8"`：拉取官方原版 `Comfy-Org/Qwen-Image-2.1` INT8-ConvRot
DIT_BASE_MODE = "official_qwen21_int8"  # @param ["official_qwen21_int8", "official_base_int8"]

# 1. 解压预置加速包或拉取最新 ComfyUI
bundle = DRIVE_HUB / "ComfyUI_qwenimage21_bundle.tar.gz"
if not COMFY_DIR.exists():
    if bundle.exists():
        print(f"🚀 从 Drive 极速解压 ComfyUI 预置包: {bundle.name} ...")
        subprocess.run(["tar", "-xzf", str(bundle), "-C", "/content/"], check=True)
    else:
        print("📥 克隆官方最新 ComfyUI...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)], check=True)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

print("⚡ 安装/校验 ComfyUI 与评测依赖 (lpips, pyarrow, safetensors)...")
subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=False)
subprocess.run(["uv", "pip", "install", "--system", "-q", "lpips", "pyarrow", "safetensors", "huggingface_hub"], check=True)

if shutil.which("aria2c") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=False)


def is_valid_safetensors(p: Path, min_mb: float = 5.0) -> bool:
    if not p.exists() or p.stat().st_size < min_mb * 1024 * 1024:
        return False
    try:
        with open(p, "rb") as f:
            n = struct.unpack("<Q", f.read(8))[0]
            if n <= 0 or n > 100 * 1024 * 1024:
                return False
            hdr = json.loads(f.read(n))
        max_end = max((v["data_offsets"][1] for k, v in hdr.items() if k != "__metadata__"), default=0)
        return (8 + n + max_end) == p.stat().st_size
    except Exception:
        return False


def stage_model(rel_sub: str, fname: str, hf_url: str, min_mb: float, save_to_drive: bool = True) -> Path:
    local_p = LOCAL_MODELS / rel_sub / fname
    local_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(local_p, min_mb):
        print(f"  ✅ NVMe 已就绪: {fname} ({local_p.stat().st_size/1e9:.2f} GB)")
        return local_p

    drive_p = DRIVE_HUB / "models" / rel_sub / fname
    drive_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(drive_p, min_mb):
        print(f"  ⚡ 从 Drive 顺序预热至本地 NVMe: {fname} ...")
        shutil.copyfile(drive_p, local_p)
        return local_p

    print(f"  ⬇️ 本地 NVMe 高速直拉: {fname} ...")
    tmp_dl = local_p.with_suffix(".dl")
    subprocess.run([
        "aria2c", "--console-log-level=warn", "--file-allocation=none",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        hf_url, "-d", str(local_p.parent), "-o", tmp_dl.name,
    ], check=True)
    os.replace(tmp_dl, local_p)
    assert is_valid_safetensors(local_p, min_mb), f"下载文件头校验失败: {local_p}"
    if save_to_drive:
        shutil.copyfile(local_p, drive_p)
    return local_p


if DIT_BASE_MODE == "official_qwen21_int8":
    DIT_PATH = stage_model(
        "diffusion_models",
        "qwen_image_2.1_int8_convrot.safetensors",
        "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        min_mb=5500,
    )
else:
    DIT_PATH = stage_model(
        "diffusion_models",
        "qwen_image_2.1_int8_convrot.safetensors",
        "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        min_mb=5500,
        save_to_drive=False,
    )

TE_PATH = stage_model(
    "text_encoders",
    "qwen3vl_8b_int8_convrot.safetensors",
    "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
    min_mb=7500,
)

# Teacher VAE 严格固定采用官方原版 Qwen-Image 2.1 BF16 VAE
VAE_PATH = stage_model(
    "vae",
    "qwen_image_2.1_vae_bf16.safetensors",
    "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
    min_mb=450,
)

LORA_8STEP_PATH = stage_model(
    "loras",
    "p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors",
    "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/loras/p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors",
    min_mb=200,
)

# 注入 ComfyUI 与本插件路径
for p in (str(COMFY_DIR), "/content"):
    if p not in sys.path:
        sys.path.insert(0, p)

# 注册本地 NVMe 搜索路径供 ComfyUI 加载器直接命中
import folder_paths
for k, sub in [
    ("diffusion_models", "diffusion_models"),
    ("text_encoders", "text_encoders"),
    ("vae", "vae"),
    ("loras", "loras"),
]:
    folder_paths.add_model_folder_path(k, str(LOCAL_MODELS / sub))

print(f"\n🎉 环境与 NVMe 模型预热全部完成！总耗时: {time.time() - t0:.1f} 秒")

In [ ]:
# @title 3. 构建四大领域混合校准图集并离线预编码缓存（编码后彻底卸载 10GB 文本塔）
import gc
import io
import json
import math
import random
import time
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image, ImageDraw, ImageFont

import comfy.model_management
import comfy.sd
import comfy.utils

CACHE_ROOT = Path("/content/qwen21_ph_cache")
IMG_DIR = CACHE_ROOT / "images"
LAT_DIR = CACHE_ROOT / "latents"
COND_DIR = CACHE_ROOT / "conds"
MANIFEST_PATH = CACHE_ROOT / "manifest.json"
for d in (IMG_DIR, LAT_DIR, COND_DIR):
    d.mkdir(parents=True, exist_ok=True)

# @markdown ### 校准集总样本数（默认 `1200` 张，分 `1120` 训练 + `80` 独立测试，缓存约需 6~9 分钟）：
TOTAL_SAMPLES = 1200  # @param {type:"integer"}
CALIB_RES = 768  # @param [768, 1024] {type:"raw"}


def center_crop_resize(im: Image.Image, size: int, mode: str = "RGBA") -> Image.Image:
    im = im.convert(mode)
    w, h = im.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    im = im.crop((left, top, left + side, top + side))
    return im.resize((size, size), Image.Resampling.LANCZOS)


def make_synthetic_cel_manga_typo(idx: int, size: int) -> tuple[Image.Image, str]:
    """生成高锐度 2D 赛璐璐色块 + 细黑线稿 + 漫画网点 + 多字号排版校准图（专治平涂断层与小字发糊）."""
    rng = random.Random(20261122 + idx)
    sub_kind = idx % 3
    if sub_kind == 0:
        # 2D 赛璐璐平涂渐变 + 锐利黑线几何人物/场景色块
        bg_c = (rng.randint(215, 250), rng.randint(220, 250), rng.randint(230, 255), 255)
        im = Image.new("RGBA", (size, size), bg_c)
        dr = ImageDraw.Draw(im)
        for _ in range(14):
            x0, y0 = rng.randint(0, size - 120), rng.randint(0, size - 120)
            x1, y1 = x0 + rng.randint(80, 360), y0 + rng.randint(80, 360)
            fill = (rng.randint(30, 245), rng.randint(30, 245), rng.randint(30, 245), 255)
            outline = (18, 18, 24, 255)
            w_line = rng.choice([2, 3, 4])
            if rng.random() < 0.5:
                dr.ellipse([x0, y0, x1, y1], fill=fill, outline=outline, width=w_line)
            else:
                dr.rounded_rectangle([x0, y0, x1, y1], radius=18, fill=fill, outline=outline, width=w_line)
        return im, "anime_cel"
    elif sub_kind == 1:
        # 黑白漫画线条 + 高频网点 (Screentone) + 对话框
        arr = np.full((size, size, 4), 248, dtype=np.uint8)
        arr[..., 3] = 255
        # 构造真实高频网点区
        yy, xx = np.mgrid[0:size, 0:size]
        period = rng.choice([4, 6, 8])
        dots = (((xx % period == 0) & (yy % period == 0))).astype(np.uint8)
        arr[size // 6 : size // 2, size // 6 : size * 5 // 6, :3] -= dots[size // 6 : size // 2, size // 6 : size * 5 // 6, None] * 170
        im = Image.fromarray(arr, mode="RGBA")
        dr = ImageDraw.Draw(im)
        for _ in range(35):
            p0 = (rng.randint(0, size), rng.randint(0, size))
            p1 = (rng.randint(0, size), rng.randint(0, size))
            dr.line([p0, p1], fill=(12, 12, 12, 255), width=rng.choice([1, 2, 3]))
        return im, "manga_line"
    else:
        # 高对比度多字号海报与锐利几何排版
        dark = rng.random() < 0.5
        bg = (18, 20, 28, 255) if dark else (246, 244, 238, 255)
        fg = (242, 245, 250, 255) if dark else (20, 22, 28, 255)
        accent = (rng.randint(180, 255), rng.randint(60, 190), rng.randint(50, 220), 255)
        im = Image.new("RGBA", (size, size), bg)
        dr = ImageDraw.Draw(im)
        dr.rectangle([36, 36, size - 36, size - 36], outline=accent, width=4)
        sample_lines = [
            "QWEN-IMAGE 2.1 PIXEL HEAD",
            "FastFrames Zero-VAE Readout 2026",
            "0123456789  PSNR 35.8 dB  5.2 ms",
            "Sharp Edges · Clean Cel · Fine Detail",
            "abcdefghijklmnopqrstuvwxyz ABCDEFG",
        ]
        y_cur = 75
        for li, text in enumerate(sample_lines):
            fsize = [42, 28, 24, 20, 16][li % 5]
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", fsize)
            except Exception:
                font = ImageFont.load_default()
            dr.text((64, y_cur), text, fill=fg if li % 2 == 0 else accent, font=font)
            y_cur += fsize + 36
        return im, "typography"


def make_synthetic_rgba_cutout(base_rgb: Image.Image, idx: int, size: int) -> Image.Image:
    """用真实图像主体配合清晰与柔化双形态 Alpha 蒙版构造原生 4 通道 RGBA 样本."""
    rng = random.Random(99000 + idx)
    rgba = base_rgb.convert("RGBA")
    mask = Image.new("L", (size, size), 0)
    dr = ImageDraw.Draw(mask)
    pad = rng.randint(48, size // 4)
    if idx % 2 == 0:
        dr.ellipse([pad, pad, size - pad, size - pad], fill=255)
    else:
        dr.rounded_rectangle([pad, pad, size - pad, size - pad], radius=64, fill=255)
    rgba.putalpha(mask)
    # 透明区 RGB 置零或保持干净边缘，与 Qwen-Image 2.1 RGBA 约定一致
    arr = np.array(rgba)
    arr[arr[..., 3] == 0, :3] = 0
    return Image.fromarray(arr, mode="RGBA")


def fetch_hf_parquet_images(dataset_id: str, max_count: int, domain_tag: str, start_idx: int) -> list[dict]:
    items = []
    shard_urls = []
    try:
        pq_api = f"https://huggingface.co/api/datasets/{dataset_id}/parquet"
        req = urllib.request.Request(pq_api, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            pq_tree = json.loads(r.read().decode("utf-8"))
        for cfg_map in pq_tree.values():
            if isinstance(cfg_map, dict):
                for split_list in cfg_map.values():
                    if isinstance(split_list, list):
                        shard_urls.extend(split_list)
    except Exception:
        pass

    if not shard_urls:
        api_url = f"https://huggingface.co/api/datasets/{dataset_id}"
        req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            info = json.loads(r.read().decode("utf-8"))
        for s in info.get("siblings", []):
            rf = s.get("rfilename", "")
            if rf.endswith(".parquet"):
                shard_urls.append(f"https://huggingface.co/datasets/{dataset_id}/resolve/main/{urllib.parse.quote(rf)}")

    for idx_sh, sh_url in enumerate(shard_urls):
        if len(items) >= max_count:
            break
        local_pq = CACHE_ROOT / f"_tmp_{dataset_id.replace('/', '_')}_{idx_sh:03d}.parquet"
        if not local_pq.exists():
            print(f"  ⬇️ 拉取公开无门控校准分片 [{idx_sh}]: {dataset_id} ...")
            urllib.request.urlretrieve(sh_url, str(local_pq))
        table = pq.read_table(local_pq)
        cols = table.column_names
        img_col = next((c for c in cols if "image" in c.lower()), None)
        if img_col is None:
            local_pq.unlink(missing_ok=True)
            continue
        col_data = table[img_col].to_pylist()
        for row_val in col_data:
            if len(items) >= max_count:
                break
            raw_bytes = row_val.get("bytes") if isinstance(row_val, dict) else None
            if not raw_bytes:
                continue
            try:
                im = Image.open(io.BytesIO(raw_bytes))
                if min(im.size) < 384:
                    continue
                im_sq = center_crop_resize(im, CALIB_RES, mode="RGBA")
                cur_id = start_idx + len(items)
                fn = f"{cur_id:05d}_{domain_tag}.png"
                im_sq.save(IMG_DIR / fn)
                items.append({"id": cur_id, "image": fn, "domain": domain_tag})
            except Exception:
                continue
        local_pq.unlink(missing_ok=True)
    return items


if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"⏭️ 检测到已完成预编码的本地缓存 ({len(manifest)} 条样本)，直接复用！")
else:
    t_c = time.time()
    n_art = int(TOTAL_SAMPLES * 0.40)
    n_photo = int(TOTAL_SAMPLES * 0.35)
    n_syn = int(TOTAL_SAMPLES * 0.15)
    n_rgba = TOTAL_SAMPLES - n_art - n_photo - n_syn

    rows = []
    print(f"📦 [1/3] 构建四大领域校准图像 ({TOTAL_SAMPLES} 张 @ {CALIB_RES}x{CALIB_RES})...")
    rows.extend(fetch_hf_parquet_images("huggan/wikiart", n_art, "art_illust", len(rows)))
    rows.extend(fetch_hf_parquet_images("detection-datasets/coco", n_photo, "real_photo", len(rows)))

    # 补充高锐度 2D 赛璐璐 / 漫画网点 / 海报排版样本
    for k in range(n_syn):
        cur_id = len(rows)
        im_s, sub_tag = make_synthetic_cel_manga_typo(k, CALIB_RES)
        fn = f"{cur_id:05d}_{sub_tag}.png"
        im_s.save(IMG_DIR / fn)
        rows.append({"id": cur_id, "image": fn, "domain": sub_tag})

    # 补充原生 4 通道 RGBA 透明贴纸样本
    base_pool = [r for r in rows if r["domain"] in ("art_illust", "real_photo", "anime_cel")]
    for k in range(n_rgba):
        cur_id = len(rows)
        src_im = Image.open(IMG_DIR / base_pool[k % len(base_pool)]["image"])
        im_rgba = make_synthetic_rgba_cutout(src_im, k, CALIB_RES)
        fn = f"{cur_id:05d}_rgba_cutout.png"
        im_rgba.save(IMG_DIR / fn)
        rows.append({"id": cur_id, "image": fn, "domain": "rgba_cutout"})

    # 固定划分 train / test（分层抽样 80 张留出集，涵盖全部细分领域）
    rng_split = random.Random(42)
    indices = list(range(len(rows)))
    rng_split.shuffle(indices)
    test_set = set(indices[: min(80, max(20, len(rows) // 12))])
    for idx_r, r in enumerate(rows):
        r["split"] = "test" if idx_r in test_set else "train"

    # 2. 单独加载 Teacher VAE 批量预编码 64 通道潜变量 x0_raw，完成后立即释放显存
    print(f"🎨 [2/3] 使用 Teacher VAE ({VAE_PATH.name}) 预编码 {len(rows)} 张 64ch 潜变量...")
    vae_sd = comfy.utils.load_torch_file(str(VAE_PATH))
    vae = comfy.sd.VAE(sd=vae_sd)
    del vae_sd

    for idx_r, r in enumerate(rows):
        im = Image.open(IMG_DIR / r["image"]).convert("RGBA")
        arr = torch.from_numpy(np.array(im)).float().div_(255.0).unsqueeze(0)  # [1, H, W, 4]
        with torch.no_grad():
            z = vae.encode(arr)  # [1, 64, H/16, W/16]
        lat_fn = f"{r['id']:05d}.pt"
        torch.save(z.to(torch.bfloat16).cpu(), LAT_DIR / lat_fn)
        r["latent"] = lat_fn
        if (idx_r + 1) % 200 == 0 or (idx_r + 1) == len(rows):
            print(f"    已编码 Latent: {idx_r + 1}/{len(rows)}")

    del vae
    comfy.model_management.cleanup_models_gc()
    gc.collect()
    torch.cuda.empty_cache()

    # 3. 单独加载 Qwen3-VL-8B 预编码 16 条跨领域条件向量，随后彻底从显存与内存卸载！
    print("🧠 [3/3] 预编码条件嵌入并彻底卸载 10GB 文本编码器...")
    PROMPTS = {
        "art_illust": [
            "High-resolution illustration with expressive brushwork, clean color transitions, rich artistic detail.",
            "2D anime illustration, crisp cel shading, clean linework, balanced color palette.",
            "Detailed painting with fine textures, subtle tonal gradients, sharp focal subject.",
        ],
        "real_photo": [
            "Sharp high-resolution photograph, natural lighting, realistic surface textures and depth of field.",
            "Candid outdoor and indoor photography, accurate colors, crisp edges and fine details.",
            "Detailed close-up and medium shot photograph with clean shadows and highlights.",
        ],
        "anime_cel": [
            "Clean 2D cel-shaded anime art with bold black ink outlines and smooth flat color blocks.",
        ],
        "manga_line": [
            "Monochrome manga page with crisp black ink lineart and fine halftone screentone patterns.",
        ],
        "typography": [
            "Graphic poster design featuring razor-sharp typography, crisp lettering, and geometric borders.",
        ],
        "rgba_cutout": [
            "Clean isolated sticker cutout on transparent alpha background with crisp silhouette edges.",
        ],
    }
    clip = comfy.sd.load_clip(
        ckpt_paths=[str(TE_PATH)],
        embedding_directory=None,
        clip_type=comfy.sd.CLIPType.QWEN_IMAGE,
    )
    cond_map = {}
    cid_counter = 0
    for dom, plist in PROMPTS.items():
        cond_map[dom] = []
        for ptxt in plist:
            tokens = clip.tokenize(ptxt, images=[], keep_vision=True, prevent_empty_text=True)
            cond = clip.encode_from_tokens_scheduled(tokens)
            cross_attn = cond[0][0].detach().to(torch.bfloat16).cpu()
            cfn = f"cond_{cid_counter:03d}.pt"
            torch.save(cross_attn, COND_DIR / cfn)
            cond_map[dom].append(cfn)
            cid_counter += 1

    for r in rows:
        opts = cond_map.get(r["domain"], cond_map["real_photo"])
        r["cond"] = opts[r["id"] % len(opts)]

    del clip
    comfy.model_management.unload_all_models()
    comfy.model_management.soft_empty_cache()
    gc.collect()
    torch.cuda.empty_cache()

    MANIFEST_PATH.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    manifest = rows
    print(f"✅ 预编码全部完成！耗时 {time.time() - t_c:.1f}s | 当前空闲显存: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
# @title 4. P0 烟雾门禁验证 + 3000 Rollouts 双模式交替正式训练（自动存盘回 Google Drive）
from collections import deque
import gc
import json
import math
import random
import shutil
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

import comfy.model_management
import comfy.sd
import comfy.utils
import importlib.util

# 支持刷新页面后无需重跑 Cell 2 直接衔接已有本地 NVMe 权重
LOCAL_MODELS = Path("/content/local_models")
if "DIT_PATH" not in globals():
    DIT_PATH = LOCAL_MODELS / "diffusion_models/qwen_image_2.1_int8_convrot.safetensors"
if "VAE_PATH" not in globals():
    VAE_PATH = LOCAL_MODELS / "vae/qwen_image_2.1_vae_bf16.safetensors"
if "LORA_8STEP_PATH" not in globals():
    LORA_8STEP_PATH = LOCAL_MODELS / "loras/p_q_image_2.1_8_steps_v0.1_comfyui_bf16.safetensors"

spec = importlib.util.spec_from_file_location("qwen21_pixel_head", "/content/ComfyUI-QwenImage21-PixelHead/qwen21_pixel_head.py")
ph_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ph_mod)
QwenImage21PixelHead = ph_mod.QwenImage21PixelHead
save_qwen21_pixel_head = ph_mod.save_qwen21_pixel_head
load_qwen21_pixel_head = ph_mod.load_qwen21_pixel_head

# @markdown ### 训练超参数（L4 24GB 黄金默认值，去除了拖慢步速的训练期 VGG-LPIPS，3000 Rollouts 约 20~28 分钟）：
TOTAL_ROLLOUTS = 3000        # @param {type:"integer"}
REPLAY_BUFFER_SIZE = 64      # @param {type:"integer"}
UPDATES_PER_ROLLOUT = 16     # @param {type:"integer"}
WINDOW_TOKENS = 32           # @param {type:"integer"}
MARGIN_TOKENS = 2            # @param {type:"integer"}
HEAD_WIDTH = 1024            # @param [768, 1024] {type:"raw"}
LEARNING_RATE = 3e-4         # @param {type:"number"}
WARMUP_STEPS = 200           # @param {type:"integer"}
SAVE_EVERY_ROLLOUTS = 500    # @param {type:"integer"}
RESUME_IF_EXISTS = True      # @param {type:"boolean"}

DEVICE = torch.device("cuda")
CKPT_DRIVE = Path("/content/drive/MyDrive/FastFrames_QwenImage21/checkpoints")
PIXEL_HEADS_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads")
VAE_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae")

manifest = json.loads(Path("/content/qwen21_ph_cache/manifest.json").read_text(encoding="utf-8"))
train_rows = [r for r in manifest if r["split"] == "train"]
test_rows = [r for r in manifest if r["split"] == "test"]
# 固定打乱 train_rows，确保艺术插画、真实摄影、2D 赛璐璐线稿排版与 RGBA 贴纸从第 1 步起均匀交错
random.Random(20261122).shuffle(train_rows)
print(f"📊 训练集: {len(train_rows)} 张（四领域已均匀混洗） | 独立验证集: {len(test_rows)} 张")

# 1. 加载冻结的 Qwen-Image 2.1 Base DiT、8-Step Turbo LoRA 分支与 Teacher VAE
comfy.model_management.unload_all_models()
gc.collect()
torch.cuda.empty_cache()

print("📦 加载冻结的 Qwen-Image 2.1 DiT 与 8-Step LoRA...")
model_base = comfy.sd.load_diffusion_model(str(DIT_PATH))
lora_sd = comfy.utils.load_torch_file(str(LORA_8STEP_PATH), safe_load=True)
model_turbo, _ = comfy.sd.load_lora_for_models(model_base, None, lora_sd, 1.0, 0.0)
del lora_sd

vae_sd = comfy.utils.load_torch_file(str(VAE_PATH))
teacher_vae = comfy.sd.VAE(sd=vae_sd)
del vae_sd

# 获取精确的流偏移时间表：
# - Turbo (8 steps simple, shift=0.69): 末步主要分布在 [0.11, 0.30]（中心 8步 ≈ 0.2217, 4步 ≈ 0.399, 12步 ≈ 0.153）
# - Base  (18~35 steps simple, shift=0.69): 末步主要分布在 [0.038, 0.115]
TURBO_SIGMA_RANGE = (0.110, 0.300)
BASE_SIGMA_RANGE = (0.038, 0.115)


class Qwen21RolloutEngine:
    """单步加噪前向截获器：无需多步循环，1 次 DiT 前向即可提取真实末步 (f, xt_norm, VAE(x0_hat))."""

    def __init__(self, patcher_base, patcher_turbo, vae, cache_dir: Path):
        self.patcher_base = patcher_base
        self.patcher_turbo = patcher_turbo
        self.vae = vae
        self.lat_dir = cache_dir / "latents"
        self.cond_dir = cache_dir / "conds"
        self.cond_mem = {}

    def _get_cond(self, fn: str) -> torch.Tensor:
        if fn not in self.cond_mem:
            self.cond_mem[fn] = torch.load(self.cond_dir / fn, map_location="cpu", weights_only=True)
        return self.cond_mem[fn]

    @torch.no_grad()
    def rollout(self, row: dict, sigma: float, use_turbo: bool, seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        patcher = self.patcher_turbo if use_turbo else self.patcher_base
        comfy.model_management.load_models_gpu([patcher])
        bmodel = patcher.model
        lat_fmt = bmodel.latent_format

        x0_raw = torch.load(self.lat_dir / row["latent"], map_location=DEVICE, weights_only=True).float()
        x0_norm = lat_fmt.process_in(x0_raw)
        cond = self._get_cond(row["cond"]).to(device=DEVICE, dtype=bmodel.get_dtype_inference())

        gen = torch.Generator(device=DEVICE).manual_seed(int(seed))
        noise = torch.randn(x0_norm.shape, generator=gen, device=DEVICE, dtype=torch.float32)
        xt_norm = (1.0 - sigma) * x0_norm + sigma * noise
        sig_t = torch.tensor([sigma], device=DEVICE, dtype=torch.float32)

        captured = {}

        def _hook(_mod, inputs):
            if "f" not in captured:
                captured["f"] = inputs[0].detach()

        if hasattr(bmodel.diffusion_model, "reset_prefix_cache"):
            bmodel.diffusion_model.reset_prefix_cache(False)

        handle = bmodel.diffusion_model.proj_out.register_forward_pre_hook(_hook)
        try:
            # ComfyUI BaseModel.apply_model already returns calculate_denoised(sigma, model_output, xt_norm) = x0_hat_norm!
            x0_hat_norm = bmodel.apply_model(xt_norm, sig_t, c_crossattn=cond).float()
        finally:
            handle.remove()

        f = captured["f"].float()  # [1, gh*gw, 4096]
        x0_hat_raw = lat_fmt.process_out(x0_hat_norm)

        # Teacher VAE 解码得到 [1, H, W, 4] in [0, 1] -> 转为 [-1, 1] NCHW
        rgba_01 = self.vae.decode(x0_hat_raw.cpu())  # [1, H, W, C] on CPU
        if rgba_01.shape[-1] == 3:
            alpha = torch.ones((*rgba_01.shape[:3], 1), dtype=rgba_01.dtype)
            rgba_01 = torch.cat([rgba_01, alpha], dim=-1)
        tgt_nchw = (rgba_01[0].permute(2, 0, 1).float() * 2.0 - 1.0).clamp_(-1.0, 1.0)  # [4, H, W]

        gh, gw = x0_hat_norm.shape[-2], x0_hat_norm.shape[-1]
        f_grid = f[0].view(gh, gw, -1).to(dtype=torch.bfloat16, device="cpu")
        x0_hat_cpu = x0_hat_norm[0].to(dtype=torch.bfloat16, device="cpu")
        return f_grid, x0_hat_cpu, tgt_nchw.to(dtype=torch.bfloat16, device="cpu")


engine = Qwen21RolloutEngine(model_base, model_turbo, teacher_vae, Path("/content/qwen21_ph_cache"))


def pixel_head_loss(pred: torch.Tensor, tgt: torch.Tensor) -> tuple[torch.Tensor, float]:
    """严格对齐 Apple FastFrames 原版重构损失 (0.5 L1 + 5.0 MSE + 0.05 Grad) + Alpha 通道回归."""
    pred_rgb, tgt_rgb = pred[:, :3], tgt[:, :3]
    l1 = (pred_rgb - tgt_rgb).abs().mean()
    mse = ((pred_rgb - tgt_rgb) ** 2).mean()
    gh = (pred_rgb.diff(dim=-2) - tgt_rgb.diff(dim=-2)).abs().mean()
    gw = (pred_rgb.diff(dim=-1) - tgt_rgb.diff(dim=-1)).abs().mean()

    pred_a, tgt_a = pred[:, 3:], tgt[:, 3:]
    alpha_l1 = (pred_a - tgt_a).abs().mean()
    alpha_mse = ((pred_a - tgt_a) ** 2).mean()
    alpha_grad = (pred_a.diff(dim=-2) - tgt_a.diff(dim=-2)).abs().mean() + (pred_a.diff(dim=-1) - tgt_a.diff(dim=-1)).abs().mean()

    total = 0.5 * l1 + 5.0 * mse + 0.05 * (gh + gw) + 0.25 * alpha_l1 + 2.5 * alpha_mse + 0.025 * alpha_grad
    # PSNR in [0, 1] RGB space: MSE_[0,1] = MSE_[-1,1] / 4
    mse_01 = max(1e-10, float(mse.detach().item()) * 0.25)
    psnr_db = -10.0 * math.log10(mse_01)
    return total, psnr_db


def random_token_window(f_grid, xt_c, tgt_c, win_tok: int, mar_tok: int, rng: random.Random):
    gh, gw = f_grid.shape[:2]
    span = min(gh, gw, win_tok + 2 * mar_tok)
    y0 = rng.randint(0, gh - span)
    x0 = rng.randint(0, gw - span)
    mt = 0 if y0 == 0 else mar_tok
    mb = 0 if (y0 + span) == gh else mar_tok
    ml = 0 if x0 == 0 else mar_tok
    mr = 0 if (x0 + span) == gw else mar_tok

    f_sub = f_grid[y0 : y0 + span, x0 : x0 + span].unsqueeze(0)
    x_sub = xt_c[:, y0 : y0 + span, x0 : x0 + span].unsqueeze(0)
    t_sub = tgt_c[:, y0 * 16 : (y0 + span) * 16, x0 * 16 : (x0 + span) * 16].unsqueeze(0)
    return f_sub, x_sub, t_sub, (mt * 16, mb * 16, ml * 16, mr * 16)


def strip_px_margin(img: torch.Tensor, mar_px: tuple[int, int, int, int]) -> torch.Tensor:
    mt, mb, ml, mr = mar_px
    h, w = img.shape[-2], img.shape[-1]
    return img[..., mt : h - mb, ml : w - mr]


# ==============================================================================
# Stage 4A: 极速 P0 烟雾门禁（校验特征维度、数值无 NaN，以及前 100 步从 ~10dB 爬升至 >=18.5dB）
# 注：全新随机初始化的 5 层 ConvNeXt+PixelShuffle 头在 100 步正常处于 20~23 dB 区间，
#     需经 Stage 4B 的 48,000 次梯度更新收敛至 33~36+ dB。
# ==============================================================================
latest_ckpt = CKPT_DRIVE / "latest_state.pt"
if not (RESUME_IF_EXISTS and latest_ckpt.exists()):
    print("\n🔬 [Stage 4A] 运行 5 秒 P0 烟雾健康度门禁 (2 个固定 24x24 窗口 × 100 步)...")
    smoke_rng = random.Random(7)
    smoke_wins = []
    for i in range(2):
        f_g, x_g, t_g = engine.rollout(train_rows[i], 0.2217, True, 1000 + i)
        assert f_g.ndim == 3 and f_g.shape[-1] == 4096, f"特征维度异常: {f_g.shape}"
        assert x_g.shape[0] == 64 and x_g.shape[1:] == f_g.shape[:2], f"潜变量网格异常: {x_g.shape} vs {f_g.shape}"
        f_s, x_s, t_s, mp = random_token_window(f_g, x_g, t_g, 24, 2, smoke_rng)
        smoke_wins.append((
            f_s.to(DEVICE, torch.bfloat16),
            x_s.to(DEVICE, torch.bfloat16),
            strip_px_margin(t_s.to(DEVICE, torch.float32), mp),
            mp,
        ))
    smoke_head = QwenImage21PixelHead(width=512, depth_low=2, depth_high=1, out_channels=4).to(DEVICE)
    smoke_opt = torch.optim.AdamW(smoke_head.parameters(), lr=8e-4, betas=(0.9, 0.99), weight_decay=0.0)
    init_psnrs, last_psnrs = [], []
    for st in range(1, 101):
        f_s, x_s, t_crop, mp = smoke_wins[st % 2]
        smoke_opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = smoke_head(f_s, x_s)
        loss, psnr_v = pixel_head_loss(strip_px_margin(pred, mp), t_crop)
        assert math.isfinite(float(loss.item())), "检测到 NaN/Inf Loss！"
        loss.backward()
        torch.nn.utils.clip_grad_norm_(smoke_head.parameters(), 2.0)
        smoke_opt.step()
        if st <= 4:
            init_psnrs.append(psnr_v)
        if st in (1, 50, 100):
            print(f"    [P0 step {st:3d}/100] loss={loss.item():.4f} | window_PSNR={psnr_v:.2f} dB")
        if st > 85:
            last_psnrs.append(psnr_v)
    p0_start = sum(init_psnrs) / len(init_psnrs)
    mean_smoke_psnr = sum(last_psnrs) / len(last_psnrs)
    del smoke_head, smoke_opt, smoke_wins
    torch.cuda.empty_cache()
    print(f"  ✅ P0 门禁通过！PSNR 从初始 {p0_start:.2f} dB 极速跃升至 {mean_smoke_psnr:.2f} dB (+{mean_smoke_psnr - p0_start:.2f} dB)")
    assert mean_smoke_psnr >= 18.5 and (mean_smoke_psnr - p0_start) >= 5.0, (
        f"P0 烟雾测试收敛异常 ({p0_start:.2f} -> {mean_smoke_psnr:.2f} dB)！"
    )

# ==============================================================================
# Stage 4B: 正式 Replay-Buffer 双模式交替训练（带线性 Warmup + 余弦退火）
# ==============================================================================
head = QwenImage21PixelHead(width=HEAD_WIDTH, depth_low=2, depth_high=1, out_channels=4).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f"\n🚀 [Stage 4B] 初始化正式 QwenImage21PixelHead: width={HEAD_WIDTH}, 参数量={n_params/1e6:.2f}M (~{n_params*4/(1024*1024):.1f} MiB FP32)")

total_grad_steps = TOTAL_ROLLOUTS * UPDATES_PER_ROLLOUT
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=1e-2, betas=(0.9, 0.99))


def set_step_lr(opt, cur_step: int) -> float:
    warm = min(1.0, (cur_step + 1) / max(1, WARMUP_STEPS))
    prog = min(1.0, cur_step / max(1, total_grad_steps))
    cos_factor = 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog))
    lr_now = LEARNING_RATE * warm * cos_factor
    for pg in opt.param_groups:
        pg["lr"] = lr_now
    return lr_now


start_rollout = 0
opt_step_count = 0
best_eval_psnr = 0.0
history = []

if RESUME_IF_EXISTS and latest_ckpt.exists():
    st_dict = torch.load(latest_ckpt, map_location=DEVICE, weights_only=False)
    head.load_state_dict(st_dict["head"])
    optimizer.load_state_dict(st_dict["optimizer"])
    start_rollout = int(st_dict["rollout"])
    opt_step_count = int(st_dict.get("opt_step_count", start_rollout * UPDATES_PER_ROLLOUT))
    best_eval_psnr = float(st_dict.get("best_eval_psnr", 0.0))
    history = list(st_dict.get("history", []))
    print(f"🔄 从断点恢复继续训练: Rollout {start_rollout}/{TOTAL_ROLLOUTS} (历史最优验证 PSNR={best_eval_psnr:.2f} dB)")

replay = deque(maxlen=REPLAY_BUFFER_SIZE)
rng = random.Random(20261122 + start_rollout)


@torch.no_grad()
def evaluate_heldout(n_eval: int = 24) -> dict:
    head.eval()
    psnrs_turbo, psnrs_base = [], []
    subset = test_rows[: min(n_eval, len(test_rows))]
    for idx_e, row in enumerate(subset):
        use_t = (idx_e % 2 == 0)
        sig = 0.2217 if use_t else 0.0680
        f_g, x_g, t_g = engine.rollout(row, sig, use_t, 880000 + idx_e)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_g.unsqueeze(0).to(DEVICE, torch.bfloat16), x_g.unsqueeze(0).to(DEVICE, torch.bfloat16))
        pred_rgb01 = ((pred[0, :3].float().cpu() + 1.0) * 0.5).clamp(0, 1)
        tgt_rgb01 = ((t_g[:3].float() + 1.0) * 0.5).clamp(0, 1)
        mse_v = max(1e-10, float(((pred_rgb01 - tgt_rgb01) ** 2).mean().item()))
        p_db = -10.0 * math.log10(mse_v)
        if use_t:
            psnrs_turbo.append(p_db)
        else:
            psnrs_base.append(p_db)
    head.train()
    all_p = psnrs_turbo + psnrs_base
    return {
        "mean_psnr": sum(all_p) / len(all_p),
        "turbo_psnr": sum(psnrs_turbo) / max(1, len(psnrs_turbo)),
        "base_psnr": sum(psnrs_base) / max(1, len(psnrs_base)),
    }


t_train = time.time()
running_loss, running_psnr, running_n = 0.0, 0.0, 0

for r_idx in range(start_rollout + 1, TOTAL_ROLLOUTS + 1):
    row = train_rows[(r_idx - 1) % len(train_rows)]
    # 每 8 个 Rollout 切换一次 Turbo LoRA 与 Base 分支，减少 87.5% LoRA 热插拔开销，同时保持 64 槽 Replay 内 50%/50% 均衡
    use_turbo = (((r_idx - 1) // 8) % 2 == 0)
    if use_turbo:
        sigma = 0.2217 if (r_idx % 2 == 1) else rng.uniform(*TURBO_SIGMA_RANGE)
    else:
        sigma = rng.uniform(*BASE_SIGMA_RANGE)

    f_g, x_g, t_g = engine.rollout(row, sigma, use_turbo, 100000 + r_idx)
    replay.append((f_g, x_g, t_g))

    # 在重放缓冲区做小批量窗口梯度更新
    n_upd = UPDATES_PER_ROLLOUT if len(replay) >= 8 else 4
    cur_lr = LEARNING_RATE
    for _ in range(n_upd):
        cur_lr = set_step_lr(optimizer, opt_step_count)
        bf, bx, bt = replay[rng.randrange(len(replay))]
        f_s, x_s, t_s, mp = random_token_window(bf, bx, bt, WINDOW_TOKENS, MARGIN_TOKENS, rng)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_s.to(DEVICE, torch.bfloat16), x_s.to(DEVICE, torch.bfloat16))
        loss, psnr_v = pixel_head_loss(strip_px_margin(pred, mp), strip_px_margin(t_s.to(DEVICE, torch.float32), mp))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        optimizer.step()
        opt_step_count += 1
        running_loss += float(loss.detach().item())
        running_psnr += psnr_v
        running_n += 1

    if r_idx % 50 == 0 or r_idx == 1:
        el = time.time() - t_train
        rate = (r_idx - start_rollout) / max(1e-3, el)
        eta_min = (TOTAL_ROLLOUTS - r_idx) / max(1e-5, rate) / 60.0
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        print(
            f"[Rollout {r_idx:4d}/{TOTAL_ROLLOUTS}] "
            f"loss={running_loss/max(1,running_n):.4f} | train_PSNR={running_psnr/max(1,running_n):.2f} dB | "
            f"lr={cur_lr:.2e} | VRAM={vram_gb:.1f}GB | ETA={eta_min:.1f}m"
        )
        running_loss, running_psnr, running_n = 0.0, 0.0, 0

    if r_idx % SAVE_EVERY_ROLLOUTS == 0 or r_idx == TOTAL_ROLLOUTS:
        ev = evaluate_heldout(24)
        history.append({"rollout": r_idx, **ev})
        print(
            f"  🎯 [Eval @ {r_idx}] Mean PSNR = {ev['mean_psnr']:.2f} dB "
            f"(8-step Turbo: {ev['turbo_psnr']:.2f} dB | Base 20-30s: {ev['base_psnr']:.2f} dB)"
        )
        meta_stamp = {
            "base_model": DIT_PATH.name,
            "teacher_vae": VAE_PATH.name,
            "rollouts": str(r_idx),
            "eval_psnr_db": f"{ev['mean_psnr']:.2f}",
            "sigma_ranges": "0.038-0.115,0.110-0.300",
        }
        snap_path = CKPT_DRIVE / f"qwen_image_2.1_pixel_head_r{r_idx:04d}.safetensors"
        save_qwen21_pixel_head(head, snap_path, extra_meta=meta_stamp)

        if ev["mean_psnr"] >= best_eval_psnr or r_idx == TOTAL_ROLLOUTS:
            best_eval_psnr = max(best_eval_psnr, ev["mean_psnr"])
            for dst in (
                PIXEL_HEADS_DRIVE / "qwen_image_2.1_pixel_head.safetensors",
                VAE_DRIVE / "qwen_image_2.1_pixel_head.safetensors",
                CKPT_DRIVE / "qwen_image_2.1_pixel_head_best.safetensors",
            ):
                shutil.copyfile(snap_path, dst)
            print(f"  💾 已同步最新最优权重至 ComfyUI_Hub: qwen_image_2.1_pixel_head.safetensors ({snap_path.stat().st_size/(1024*1024):.1f} MiB)")

        torch.save(
            {
                "rollout": r_idx,
                "opt_step_count": opt_step_count,
                "head": head.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best_eval_psnr": best_eval_psnr,
                "history": history,
            },
            latest_ckpt,
        )

print(f"\n🎉 正式训练全部圆满完成！最优独立测试集 PSNR = {best_eval_psnr:.2f} dB")

In [ ]:
# @title 5. 自动化三方硬核基准评测：Official VAE vs madebyollin/taeqi2_1 vs 你的 PixelHead
import math
import time
import urllib.request
from pathlib import Path
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw

import lpips
import numpy as np
import torch
import comfy.sd
import comfy.utils

lpips_vgg = lpips.LPIPS(net="vgg", verbose=False).to(DEVICE).eval()
lpips_vgg.requires_grad_(False)

EVAL_DRIVE = Path("/content/drive/MyDrive/FastFrames_QwenImage21/eval_report")
EVAL_DRIVE.mkdir(parents=True, exist_ok=True)
BEST_HEAD_PATH = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/qwen_image_2.1_pixel_head.safetensors")

# 拉取社区现有的 64ch 潜空间小解码器 madebyollin/taeqi2_1 进行公平同台打擂（若加载结构兼容则同屏跑分）
TAE_PATH = Path("/content/local_models/vae/taeqi2_1.safetensors")
if not TAE_PATH.exists():
    try:
        urllib.request.urlretrieve(
            "https://huggingface.co/madebyollin/taeqi2_1/resolve/main/taeqi2_1.safetensors",
            str(TAE_PATH),
        )
    except Exception as e:
        print("TAE 下载跳过:", e)

tae_vae = None
if TAE_PATH.exists():
    try:
        tae_vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(TAE_PATH)))
        print("✅ 已加载参照对比模型: madebyollin/taeqi2_1 (29.2 MiB)")
    except Exception as e:
        print("⚠️ 当前 ComfyUI 版本未内置 taeqi2_1 解码分支，将直接对比 Teacher VAE 与 PixelHead:", e)

eval_head = load_qwen21_pixel_head(BEST_HEAD_PATH, device=DEVICE, dtype=torch.bfloat16)

by_domain = {}
per_item_ph, per_item_tae = [], []
ph_latencies_ms, vae_latencies_ms, tae_latencies_ms = [], [], []
crop_rows = []

print("🔍 正在独立留出集上执行逐领域 PSNR / LPIPS / 延迟毫秒级实测...")
for idx_e, row in enumerate(test_rows[:40]):
    use_t = (idx_e % 2 == 0)
    sig = 0.2217 if use_t else 0.0680

    patcher = model_turbo if use_t else model_base
    comfy.model_management.load_models_gpu([patcher])
    bmodel = patcher.model
    lat_fmt = bmodel.latent_format

    x0_raw = torch.load(Path("/content/qwen21_ph_cache/latents") / row["latent"], map_location=DEVICE, weights_only=True).float()
    x0_norm = lat_fmt.process_in(x0_raw)
    cond = engine._get_cond(row["cond"]).to(device=DEVICE, dtype=bmodel.get_dtype_inference())
    gen = torch.Generator(device=DEVICE).manual_seed(990000 + idx_e)
    xt_norm = (1.0 - sig) * x0_norm + sig * torch.randn(x0_norm.shape, generator=gen, device=DEVICE)
    sig_t = torch.tensor([sig], device=DEVICE, dtype=torch.float32)

    if hasattr(bmodel.diffusion_model, "reset_prefix_cache"):
        bmodel.diffusion_model.reset_prefix_cache(False)
    cap = {}
    h_hook = bmodel.diffusion_model.proj_out.register_forward_pre_hook(lambda _m, inp: (cap.setdefault("f", inp[0].detach()), None)[1])
    with torch.no_grad():
        x0_hat_norm = bmodel.apply_model(xt_norm, sig_t, c_crossattn=cond).float()
    h_hook.remove()
    x0_hat_raw = lat_fmt.process_out(x0_hat_norm)

    # 1. 计时 Teacher VAE Decode
    torch.cuda.synchronize()
    t_v0 = time.perf_counter()
    with torch.no_grad():
        ref_nhwc = teacher_vae.decode(x0_hat_raw.cpu())[..., :3].float().clamp(0, 1)
    torch.cuda.synchronize()
    vae_latencies_ms.append((time.perf_counter() - t_v0) * 1000.0)

    # 2. 计时 PixelHead Decode（输入去噪后的 x0_hat_norm 与 4096 维末层特征）
    f_in = cap["f"].to(DEVICE, torch.bfloat16)
    x_in = x0_hat_norm.to(DEVICE, torch.bfloat16)
    torch.cuda.synchronize()
    t_p0 = time.perf_counter()
    with torch.no_grad():
        ph_nchw = eval_head(f_in, x_in)
    torch.cuda.synchronize()
    ph_latencies_ms.append((time.perf_counter() - t_p0) * 1000.0)
    ph_nhwc = ((ph_nchw[:, :3].float().cpu() + 1.0) * 0.5).clamp(0, 1).movedim(1, -1)

    ref_m11 = (ref_nhwc.movedim(-1, 1).to(DEVICE) * 2.0 - 1.0)
    ph_m11 = (ph_nhwc.movedim(-1, 1).to(DEVICE) * 2.0 - 1.0)
    mse_ph = max(1e-10, float(((ph_nhwc - ref_nhwc) ** 2).mean().item()))
    psnr_ph = -10.0 * math.log10(mse_ph)
    with torch.no_grad():
        lp_ph = float(lpips_vgg(ph_m11, ref_m11).mean().item())

    dom = row["domain"]
    by_domain.setdefault(dom, []).append((psnr_ph, lp_ph))
    per_item_ph.append((psnr_ph, lp_ph))

    # 3. 可选计时 taeqi2_1
    tae_nhwc = None
    if tae_vae is not None:
        try:
            torch.cuda.synchronize()
            t_t0 = time.perf_counter()
            with torch.no_grad():
                tae_nhwc = tae_vae.decode(x0_hat_raw.cpu())[..., :3].float().clamp(0, 1)
            torch.cuda.synchronize()
            tae_latencies_ms.append((time.perf_counter() - t_t0) * 1000.0)
            mse_t = max(1e-10, float(((tae_nhwc - ref_nhwc) ** 2).mean().item()))
            tae_m11 = (tae_nhwc.movedim(-1, 1).to(DEVICE) * 2.0 - 1.0)
            with torch.no_grad():
                lp_t = float(lpips_vgg(tae_m11, ref_m11).mean().item())
            per_item_tae.append((-10.0 * math.log10(mse_t), lp_t))
        except Exception:
            tae_vae = None

    if len(crop_rows) < 6:
        crop_rows.append((dom, ref_nhwc[0], ph_nhwc[0], tae_nhwc[0] if tae_nhwc is not None else None, psnr_ph))

mean_psnr_ph = sum(x[0] for x in per_item_ph) / len(per_item_ph)
mean_lp_ph = sum(x[1] for x in per_item_ph) / len(per_item_ph)
med_vae_ms = sorted(vae_latencies_ms)[len(vae_latencies_ms) // 2]
med_ph_ms = sorted(ph_latencies_ms)[len(ph_latencies_ms) // 2]
speedup = med_vae_ms / max(0.05, med_ph_ms)

report_lines = [
    "# Qwen-Image 2.1 Direct Pixel Head — Benchmark Report",
    "",
    f"- **Base DiT**: `{DIT_PATH.name}` + `8-Step LoRA` alternating",
    f"- **Teacher Reference VAE**: `{VAE_PATH.name}` (`644.2 MiB`)",
    f"- **Trained Pixel Head**: `qwen_image_2.1_pixel_head.safetensors` (`{BEST_HEAD_PATH.stat().st_size/(1024*1024):.1f} MiB`, RGBA 4-ch)",
    "",
    "## 1. Overall Speed & Reconstruction Fidelity (`768×768` Held-Out Split)",
    "",
    "| Decoder | File Size | Median Decode Latency | Speedup vs VAE | PSNR (↑ dB) | LPIPS-VGG (↓) | Native RGBA |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: |",
    f"| **Stock Qwen-Image 2.1 VAE** | `644.2 MiB` | `{med_vae_ms:.1f} ms` | `1.0×` | *(Reference)* | `0.0000` | ✅ |",
]
if per_item_tae:
    med_tae_ms = sorted(tae_latencies_ms)[len(tae_latencies_ms) // 2]
    m_psnr_t = sum(x[0] for x in per_item_tae) / len(per_item_tae)
    m_lp_t = sum(x[1] for x in per_item_tae) / len(per_item_tae)
    report_lines.append(
        f"| `madebyollin/taeqi2_1` (Latent-only) | `29.2 MiB` | `{med_tae_ms:.1f} ms` | `{med_vae_ms/max(0.05,med_tae_ms):.1f}×` | `{m_psnr_t:.2f} dB` | `{m_lp_t:.4f}` | ❌/Partial |"
    )
report_lines.append(
    f"| **Ours: `QwenImage21 PixelHead`** | **`{BEST_HEAD_PATH.stat().st_size/(1024*1024):.1f} MiB`** | **`{med_ph_ms:.2f} ms`** | **`{speedup:.1f}×`** | **`{mean_psnr_ph:.2f} dB`** | **`{mean_lp_ph:.4f}`** | **✅ Full 4-ch** |"
)
report_lines += [
    "",
    "## 2. Per-Domain Breakdown",
    "",
    "| Domain | Samples | PSNR (↑ dB) | LPIPS-VGG (↓) |",
    "| :--- | :---: | :---: | :---: |",
]
for dom, vals in sorted(by_domain.items()):
    report_lines.append(
        f"| `{dom}` | {len(vals)} | `{sum(v[0] for v in vals)/len(vals):.2f} dB` | `{sum(v[1] for v in vals)/len(vals):.4f}` |"
    )

report_md = "\n".join(report_lines) + "\n"
(EVAL_DRIVE / "BENCHMARK_REPORT.md").write_text(report_md, encoding="utf-8")

# 生成直观的 1:1 放大对比图板
cols = 3 if any(c[3] is not None for c in crop_rows) else 2
cell_px = 384
grid_im = Image.new("RGB", (cols * cell_px, len(crop_rows) * cell_px), (20, 20, 24))
dr_g = ImageDraw.Draw(grid_im)

for ri, (dom, ref_t, ph_t, tae_t, p_db) in enumerate(crop_rows):
    h, w, _ = ref_t.shape
    y0, x0 = (h - cell_px) // 2, (w - cell_px) // 2
    panels = [("Stock VAE (644MB)", ref_t[y0:y0+cell_px, x0:x0+cell_px])]
    if cols == 3 and tae_t is not None:
        panels.append(("taeqi2_1 (Latent)", tae_t[y0:y0+cell_px, x0:x0+cell_px]))
    panels.append((f"PixelHead ({p_db:.1f}dB, {med_ph_ms:.1f}ms)", ph_t[y0:y0+cell_px, x0:x0+cell_px]))
    for ci, (label, patch_t) in enumerate(panels):
        pim = Image.fromarray((patch_t.numpy() * 255.0).round().astype(np.uint8))
        grid_im.paste(pim, (ci * cell_px, ri * cell_px))
        dr_g.rectangle([ci * cell_px + 6, ri * cell_px + 6, ci * cell_px + 285, ri * cell_px + 30], fill=(0, 0, 0))
        dr_g.text((ci * cell_px + 12, ri * cell_px + 10), f"[{dom}] {label}", fill=(255, 255, 255))

grid_path = EVAL_DRIVE / "comparison_1to1_crops.png"
grid_im.save(grid_path)

display(Markdown(report_md))
display(IPyImage(filename=str(grid_path)))
print(f"\n✅ 完整评测报告与对比样图已保存至网盘: {EVAL_DRIVE}")

In [ ]:
# @title 6. [可选二阶王炸] 在已训好 1x 头基础上加训 `2x` 直出超分头（1024 潜空间 12ms 一步直出 2048！）
# @markdown 仅当你对上方 1x 头效果满意、想进一步产出 `qwen_image_2.1_pixel_head_x2.safetensors` 时勾选运行：
TRAIN_X2_STAGE = False  # @param {type:"boolean"}
X2_ROLLOUTS = 1500      # @param {type:"integer"}

if not TRAIN_X2_STAGE:
    print("⏭️ 当前未勾选 `TRAIN_X2_STAGE`（默认跳过）。需要训 2x 超分直出头时勾选再点运行本单元格即可！")
else:
    import random, time
    import torch
    import torch.nn.functional as F
    from pathlib import Path

    QwenImage21PixelHeadX2 = ph_mod.QwenImage21PixelHeadX2
    base_1x = load_qwen21_pixel_head(
        "/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/qwen_image_2.1_pixel_head.safetensors",
        device=DEVICE,
        dtype=torch.float32,
    )
    # 冻结 1x 主干，仅训练新增的 ~3.5M 参数二阶 PixelShuffle 超分分支（速度极快、显存极省！）
    for p in base_1x.parameters():
        p.requires_grad_(False)
    head_x2 = QwenImage21PixelHeadX2(base=base_1x, depth_x2=2, ls_init=1.0).to(DEVICE)
    opt_x2 = torch.optim.AdamW([p for p in head_x2.parameters() if p.requires_grad], lr=2e-4, weight_decay=1e-4)
    rng2 = random.Random(2026)

    print("🚀 开始训练二阶 2x 直出超分分支...")
    t_x2 = time.time()
    for r_i in range(1, X2_ROLLOUTS + 1):
        row = train_rows[(r_i * 3571) % len(train_rows)]
        use_t = (r_i % 2 == 1)
        sig = 0.2217 if use_t else rng2.uniform(*BASE_SIGMA_RANGE)
        f_g, x_g, t_g = engine.rollout(row, sig, use_t, 500000 + r_i)
        # 构造高锐度 Lanczos/Bicubic + 边缘锐化 2x 教师目标（若有原生 1536px 原图也可直接替换）
        t_1x = t_g.unsqueeze(0).to(DEVICE, torch.float32)
        t_2x = F.interpolate(t_1x, scale_factor=2.0, mode="bicubic", align_corners=False).clamp(-1, 1)

        gh, gw = f_g.shape[:2]
        span = min(gh, gw, 24)
        y0, x0 = rng2.randint(0, gh - span), rng2.randint(0, gw - span)
        f_sub = f_g[y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16)
        x_sub = x_g[:, y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16)
        tgt_sub = t_2x[:, :, y0*32:(y0+span)*32, x0*32:(x0+span)*32]

        opt_x2.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred_2x = head_x2(f_sub, x_sub, scale=2)
        loss_2x, psnr_2x = pixel_head_loss(pred_2x[:, :, 32:-32, 32:-32], tgt_sub[:, :, 32:-32, 32:-32])
        loss_2x.backward()
        torch.nn.utils.clip_grad_norm_(head_x2.parameters(), 1.0)
        opt_x2.step()

        if r_i % 100 == 0:
            print(f"[2x Rollout {r_i}/{X2_ROLLOUTS}] loss={loss_2x.item():.4f} | 2x_PSNR={psnr_2x:.2f} dB | {time.time()-t_x2:.0f}s")

    out_x2 = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/qwen_image_2.1_pixel_head_x2.safetensors")
    save_qwen21_pixel_head(head_x2, out_x2, extra_meta={"scale": "2", "rollouts_x2": str(X2_ROLLOUTS)})
    shutil.copyfile(out_x2, Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae/qwen_image_2.1_pixel_head_x2.safetensors"))
    print(f"🎉 2x 超分直出头已保存: {out_x2} ({out_x2.stat().st_size/(1024*1024):.1f} MiB)")